# RAG Evaluation Test Set Generation

This example shows how to use the [Ragas](https://docs.ragas.io/en/stable/) (```v 0.1.22```) framework to generate a **test set** that can be used to evaluate the quality of a RAG pipeline. We then use the Python [LangChain](https://python.langchain.com/docs/introduction/) library to run some requests through this pipeline and we evaluate the quality of the results.

### <u>Requirements</u>
1. As you will accessing the LLMs and embedding models through Vector AI Engineering's Kaleidoscope Service (Vector Inference + Autoscaling), you will need to request a KScope API Key:

      Run the following command (replace ```<user_id>``` and ```<password>```) from **within the cluster** to obtain the API Key. The ```access_token``` in the output is your KScope API Key.
  ```bash
  curl -X POST -d "grant_type=password" -d "username=<user_id>" -d "password=<password>" https://kscope.vectorinstitute.ai/token
  ```
2. After obtaining the `.env` configurations, make sure to create the ```.kscope.env``` file in your home directory (```/h/<user_id>```) and set the following env variables:
- For local models through Kaleidoscope (KScope):
    ```bash
    export OPENAI_BASE_URL="https://kscope.vectorinstitute.ai/v1"
    export OPENAI_API_KEY=<kscope_api_key>
    ```
- For OpenAI models:
   ```bash
   export OPENAI_BASE_URL="https://api.openai.com/v1"
   export OPENAI_API_KEY=<openai_api_key>
   ```

## Set up the RAG workflow environment

#### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
#!pip install pdfplumber


In [3]:
#import pdfplumber

In [4]:
import numpy as np
import os
import sys

from datasets import Dataset
from pathlib import Path

from langchain.chains import RetrievalQA
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, AnswerCorrectness
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

In [5]:
cd ..

/fs01/home/ws_ikharchuk/rag_bootcamp_ik/rag_evaluation


#### Load config files

In [6]:
# Add root folder of the rag_bootcamp repo to PYTHONPATH
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))



In [7]:
from utils.load_secrets import load_env_file
load_env_file()

#### Set up some helper functions

In [8]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

#### Make sure other necessary items are in place

## Generate a sythentic test set

#### Start by loading in the documents we'll be using to augment our RAG generations

In [9]:
#load all documents

In [10]:

from langchain.document_loaders import TextLoader
from langchain.document_loaders import PyPDFLoader

In [11]:
%%time
directory_path = "/projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS"
file_list = [
   '48412CA Long-Distance Freight Trucking in Canada Industry Report.pdf',
#'48422CA Local Specialized Freight Trucking in Canada Industry Report.pdf'
]  # Replace with your actual file names

# Load only the specified files
docs = []
for file_name in file_list:
    file_path = os.path.join (directory_path, file_name)
    loader = PyPDFLoader(file_path)
    docs.extend(loader.load())  # Append loaded pages to the list
print(f"Number of source documents: {len(docs)}")
for document in docs:
    document.metadata['file_name'] = document.metadata['source']

Number of source documents: 39
CPU times: user 1.46 s, sys: 111 ms, total: 1.57 s
Wall time: 1.57 s


In [12]:
%%time
# no need 
#Process PDFs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)
chunks = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 47
CPU times: user 8.7 ms, sys: 0 ns, total: 8.7 ms
Wall time: 8.21 ms


#### Now use OpenAI to generate a test set from the data in these documents (This takes about 2-3 minutes)

**IMP Note:** The LLM and embedding model used for test set generation should be more capable than the model being evaluated. Hence, we will use OpenAI GPT-4o and OpenAI embeddings for this purpose.

Store your OpenAI API key in ```~/.ragas_openai.env``` using the following format (this is in addition to ```~/.kscope.env```):

```bash
export RAGAS_OPENAI_BASE_URL="https://api.openai.com/v1"
export RAGAS_OPENAI_API_KEY=<openai_api_key>
```

In [13]:
# Select  LLM (Llama)

In [13]:
from utils.load_secrets import load_env_file_ragas
load_env_file_ragas()

In [15]:
# GENERATOR_BASE_URL = os.environ.get("OPENAI_BASE_URL")

# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [16]:
# GENERATOR_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
# #GENERATOR_MODEL_NAME = 'DeepSeek-R1-Distill-Qwen-1.5B'
# EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

In [17]:
# llm = ChatOpenAI(
#     model=GENERATOR_MODEL_NAME,
#     temperature=0,
#     max_tokens=None,
#     base_url=GENERATOR_BASE_URL,
#     api_key=OPENAI_API_KEY
# )

In [14]:
generator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)
generator_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)

In [19]:
# model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
# encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

# print(f"Setting up the embeddings model...")
# embeddings = HuggingFaceEmbeddings(
#     model_name=   EMBEDDING_MODEL_NAME,
#     model_kwargs=model_kwargs,
#     encode_kwargs=encode_kwargs,
# )

In [15]:
def read_csv_from_directory(directory_path):
    dataframes= []
    for filename in os.listdir(directory_path):
        if filename.endswith('.csv'): 
            file_path = os.path.join(directory_path, filename)
            df = pd.read_csv(file_path)
            df["source"] = filename
            # print(df.head(1))
            dataframes.append(df)
    return pd.concat(dataframes, ignore_index=True)

In [81]:
#Load TRUCKING  data
file_paths = [#'/projects/RAG2/scotia-2/Datasets-Scotia-2/Agriculture_txt/agri_ca_co.csv', 
              '/projects/RAG2/scotia-2/Datasets-Scotia-2/Transport_txt/transport_CA.csv', 
              #'/projects/RAG2/scotia-2/Datasets-Scotia-2/Auto_txt/auto_ca.csv', 
             ]
def load_txt_file(file_path):
    # Create a TextLoader instance

    loader = TextLoader(file_path)

    # Load the document

    document = loader.load()
    chunks2 =text_splitter.split_documents(document)
    print(f"Number of text chunks: {len(chunks2)}")
    return chunks2, document

In [82]:
%%time

for file_path in file_paths:
    chunks_news, document_news = load_txt_file(file_path)


Number of text chunks: 403
CPU times: user 32 ms, sys: 4.01 ms, total: 36 ms
Wall time: 33.4 ms


In [83]:
len(document_news), len (docs)

(1, 39)

40

In [19]:
%%time
#generator_llm = (
#    model="DeepSeek-R1-Distill-Llama-8B",
#    base_url=os.environ["OPENAI_BASE_URL"],
#    api_key=os.environ["OPENAI_API_KEY"],
#)
# Define the RAG embeddings model (different than the OpenAI embedding model defined above for test set generation)
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
# print(f"Setting up the RAG LLM...")
# llm = ChatOpenAI(
#     #model="DeepSeek-R1-Distill-Llama-8B",
#     model="Meta-Llama-3.1-8B-Instruct",
#     temperature=0,
#     max_tokens=256,
#     base_url=os.environ["OPENAI_BASE_URL"],
#     api_key=os.environ["OPENAI_API_KEY"],
# )

# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-base-en-v1.5",
#     model_kwargs=model_kwargs,
#     encode_kwargs=encode_kwargs,
# )

CPU times: user 8 µs, sys: 2 µs, total: 10 µs
Wall time: 16.5 µs


In [67]:
#generator_llm.generate('what is  the day today?, Answer in no more then 10 words')

In [68]:
# %%time
# # Create generator with OpenAI model
# generator = TestsetGenerator.from_langchain(
#     generator_llm=generator_llm,
#     critic_llm=generator_llm,
#     embeddings=generator_embeddings,
# )

# # Generate the test set
# testset = generator.generate_with_langchain_docs(
#     documents=documents, 
#     test_size=1,
#     distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
# )

In [31]:
%%time
# Create generator with LLama model
# generator = TestsetGenerator.from_langchain(
#     #generator_llm=llm,
#     generator_llm=generator_llm,
#     #critic_llm=llm,
#     critic_llm=generator_llm,
#     #embeddings=embeddings,
#     embeddings = generator_embeddings
# )

# # Generate the test set
# testset = generator.generate_with_langchain_docs(
#     documents=combined_docs, 
#     test_size=50,
#     distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
# )

embedding nodes:   0%|          | 0/746 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/50 [00:00<?, ?it/s]

CPU times: user 37.9 s, sys: 1.01 s, total: 39 s
Wall time: 5min 6s


#### Save dataset

In [20]:
#testset2 = testset.to_pandas()

In [21]:
from collections import Counter

In [23]:
#len (testset2.metadata[0])

In [24]:
#testset2.tail(5)

In [25]:
import joblib

In [44]:


# Save (serialize) the object to a file
#joblib.dump(testset, "../../Testing_Data/testset_combined.pkl")


['../../Testing_Data/testset_combined.pkl']

In [45]:
#testset2.to_parquet ('../../Testing_Data/testset_combined.parquet')
#testset2.to_csv ('../../Testing_Data/testset_combined.csv', index =False)

In [26]:
#load 
# Load (deserialize) the object from file
testset = joblib.load("../../Testing_Data/testset_combined.pkl")


## Now, start the RAG pipeline!

#### Choose the RAG LLM and embedding model
Note: This is different than the OpenAI LLM and embedding model defined above for test set generation.

In [27]:
# RAG_LLM_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
# RAG_EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

Go through the embedding, storage and retrieval steps.

# Setting up having two vector stores

In [78]:
#PDF 

text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)
chunks = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 47


In [87]:
%%time
vectorstore_pdf = FAISS.from_documents(chunks, generator_embeddings)
retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k": 3})

CPU times: user 337 ms, sys: 3.97 ms, total: 341 ms
Wall time: 977 ms


In [84]:
len (chunks_news)

403

In [86]:
%%time
# news
vectorstore_news = FAISS.from_documents(chunks_news, generator_embeddings)
retriever_news = vectorstore_news.as_retriever(search_kwargs={"k": 3})

CPU times: user 2.13 s, sys: 40.3 ms, total: 2.17 s
Wall time: 5.84 s


In [88]:
from langchain.schema import BaseRetriever
from pydantic import Field
from typing import List

class CombinedRetriever(BaseRetriever):
    retrievers: List[BaseRetriever] = Field(default_factory=list)

    def get_relevant_documents(self, query: str):
        combined_results = []
        for retriever in self.retrievers:
            combined_results.extend(retriever.get_relevant_documents(query))
        return combined_results

# Instantiate the new combined retriever
combined_retriever = CombinedRetriever(retrievers=[retriever_pdf, retriever_news])


Iterate over the questions in our synthetic testset, and run them each through the RAG pipeline to see what answers get returned. (This also takes 2-3 minutes)

#### Generate answers for all the questions in our test set

In [89]:
# answer Questions

In [90]:
%%time
#USe OpenAi

dataset = testset.to_dataset()
answers = np.empty(len(dataset), dtype=object)

for index, row in enumerate(dataset):
    query = row["question"]
 
    #Run the query through the RAG pipeline
    rag_pipeline = RetrievalQA.from_llm(
        llm=generator_llm,
        retriever=combined_retriever
    )
    answer = rag_pipeline.invoke(input=query)
    answer = answer["result"]
    print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    
    # Store the result
    answers[index] = answer

Result 0
Question: What percentage of structural damage is being reported in Jasper due to the wildfires?
Answer: Authorities are reporting potential structural damage of up to 50% in Jasper due to the wildfires.

Result 1
Question: What is the expected growth range for adjusted diluted earnings per share according to CN's latest forecast?
Answer: I don't know.

Result 2
Question: What is the expected revenue growth for long-distance freight trucking services through the end of 2024?
Answer: The expected revenue growth for long-distance freight trucking services through the end of 2024 is a compound annual growth rate (CAGR) of 1.0%, reaching $31.3 billion. Additionally, revenue is anticipated to grow by 1.2% in 2024 alone.

Result 3
Question: What are the potential economic impacts of a labor stoppage at Canada's two largest railroad operators?
Answer: The potential economic impacts of a labor stoppage at Canada's two largest railroad operators, Canadian National Railway and Canadian 

Result 11
Question: What is TFI International's market capitalization as of FY 2023?
Answer: I don't know.

Result 12
Question: What factors contributed to the decline of the Toronto Stock Exchange?
Answer: Several factors contributed to the decline of the Toronto Stock Exchange, including:

1. **Profit Taking**: Investors engaged in profit taking after stocks that had performed well began to underperform, particularly in the technology sector.

2. **Disappointing Earnings**: Key companies, such as Tesla and Alphabet, reported lackluster earnings, leading investors to question the sustainability of the equity rally driven by technology and AI.

3. **Interest Rate Cuts**: While the Bank of Canada cut interest rates, which could generally support the market, the overall investor sentiment was cautious due to the broader economic context.

4. **Economic Slowdown**: Evidence of a slowdown in the domestic economy, including declining retail sales, contributed to market declines.

5. **Bond 

Result 24
Question: What role do technology improvements play in enhancing the safety and efficiency of freight trucking?
Answer: Technology improvements play a significant role in enhancing the safety and efficiency of freight trucking in several ways:

1. **Safety Enhancements**: Technology has introduced various safety measures aimed at ensuring drivers' health and reducing operational losses. For example, advancements in vehicle technology help prevent accidents by keeping vehicles on the road, controlling acceleration, and managing speed on steep grades.

2. **Data Collection and Analytics**: Companies are increasingly upgrading their fleets with sensors and other technologies to collect data on driving patterns. This data helps in identifying potential risks and preventing accidents, thus improving overall safety.

3. **Route Optimization**: Technologies such as the Internet of Things (IoT) and artificial intelligence (AI) optimize route planning, enhancing the efficiency of frei

Result 39
Question: How do transport modes and economy affect trade efficiency in N.A.?
Answer: I don't know.

Result 40
Question: What impact does the new CPKC-CSX rail corridor have on grain transport efficiency and Canadian rail labor talks?
Answer: I don't know.

Result 41
Question: What's the reasoning behind CPKC's Non-GAAP measures in financial reporting, and how do they improve earnings and liquidity assessment vs. U.S. GAAP?
Answer: CPKC presents Non-GAAP measures to provide an additional basis for evaluating underlying earnings and liquidity trends in their financial results. The reasoning behind using Non-GAAP measures is that they facilitate a multi-period assessment of long-term profitability and help in assessing future profitability. 

These measures allow management and stakeholders to analyze financial performance in a way that may highlight trends not captured by traditional U.S. GAAP metrics. Non-GAAP measures can offer insights into the company's operational efficie

Result 48
Question: What steps did the Bombers take for Indigenous reconciliation in line with EIC's model?
Answer: The Bombers participated in the National Day for Truth and Reconciliation by inviting over 1,000 Indigenous people from across Canada to attend a Canadian Football League game. This event aimed to bring attention to the need for reconciliation with Indigenous peoples and was part of a collaboration involving several CFL teams. Additionally, through the Atik Mason Indigenous Pilot Pathway, EIC invested over $2 million annually back into relationships with Indigenous communities and partners, which is expected to approach $3 million in 2024.

Result 49
Question: What prompted Mayor Salinas's emergency declaration regarding the migrant surge and its impact on border traffic?
Answer: Mayor Salinas issued an emergency declaration due to a "severe undocumented immigrant surge," which led to the temporary closure of train traffic to Mexico via the Eagle Pass gateway in Texas. Th

In [68]:
# %%time
# #Use LLama
# dataset = testset.to_dataset()
# answers = np.empty(len(dataset), dtype=object)

# for index, row in enumerate(dataset):
#     query = row["question"]
    
#     # Run the query through the RAG pipeline
#     rag_pipeline = RetrievalQA.from_llm(
#         llm=llm,
#         retriever=retriever
#     )
#     answer = rag_pipeline.invoke(input=query)
#     answer = answer["result"]
#     print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    
#     # Store the result
#     answers[index] = answer

Add the list of answers into our original dataset. Now we have a complete test set that is ready for evaluation.

In [91]:
dataset = dataset.add_column("answer", answers)

In [92]:
type(dataset)

datasets.arrow_dataset.Dataset

## Evaluate the results

#### Preview the final test set

In [93]:
dataset.to_pandas().head(5)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer
0,What percentage of structural damage is being ...,"[while choking back tears. \n """"We're seein...",The context reports potentially 30% to 50% str...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,Authorities are reporting potential structural...
1,What is the expected growth range for adjusted...,[ transportation of various commodities and go...,"According to CN's latest forecast, the expecte...",simple,"[{'file_name': None, 'page': None, 'source': '...",True,I don't know.
2,What is the expected revenue growth for long-d...,[ping t o addr ess l abor short ages. A s the ...,Revenue for long-distance freight trucking ser...,simple,[{'file_name': '/projects/RAG2/scotia-2/Datase...,True,The expected revenue growth for long-distance ...
3,What are the potential economic impacts of a l...,[ binding arbitration is imposed.\n Earlier...,The potential economic impacts of a labor stop...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,The potential economic impacts of a labor stop...
4,What is the purpose of the conciliation proces...,[ions to breakthrough on these paid sick leave...,The purpose of the conciliation process in lab...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,I don't know.


Run the evaluation query to score the results. In this evaluation, we are looking at the following metrics:
- *[Faithfulness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/faithfulness.html)*: Are all the claims that are made in the answer inferred from the given context(s)?
- *[Context Precision](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_precision.html)*: Did our retriever return good results that matched the question it was being asked?
- *[Answer Correctness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/answer_correctness.html)*: Was the generated answer correct? Was it complete?

In [94]:
%%time
score = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ContextPrecision(),
        AnswerCorrectness(),
    ],
    llm=generator_llm, # Using OpenAI LLM as the evaluator
    embeddings=generator_embeddings,
)


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

CPU times: user 15.3 s, sys: 199 ms, total: 15.5 s
Wall time: 1min 41s


In [95]:
%%time
#use OpenAi
# score = evaluate(
#     dataset=dataset,
#     metrics=[
#         Faithfulness(),
#         ContextPrecision(),
#         AnswerCorrectness(),
#     ],
#     llm=generator_llm, # Using OpenAI LLM as the evaluator
#     #llm=llm, # Using Llama LLM as the evaluator
#     embeddings=embeddings,
# )

CPU times: user 10 µs, sys: 1e+03 ns, total: 11 µs
Wall time: 18.8 µs


In [96]:
score.to_pandas().isna().sum(axis =0)

question              0
contexts              0
ground_truth          0
evolution_type        0
metadata              0
episode_done          0
answer                0
faithfulness          0
context_precision     0
answer_correctness    0
dtype: int64

In [97]:
score.to_pandas().tail(2)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer,faithfulness,context_precision,answer_correctness
48,What steps did the Bombers take for Indigenous...,[ people from across\nCanada to attend the Win...,"The Bombers, in line with EIC's model, partici...",multi_context,"[{'file_name': None, 'page': None, 'source': '...",True,The Bombers participated in the National Day f...,0.833333,1.0,0.817251
49,What prompted Mayor Salinas's emergency declar...,[ late 2021 to help fund its\napproximately $3...,Mayor Salinas issued an emergency declaration ...,multi_context,"[{'file_name': None, 'page': None, 'source': '...",True,Mayor Salinas issued an emergency declaration ...,1.000000,1.0,0.612501


In [98]:
score["answer_correctness"].mean()

0.5750182259677298

In [99]:
score["faithfulness"].mean()

0.5718615107115107

In [45]:
#score.to_pandas().to_parquet  ("../../Testing_Data/score_llama_answeropenAi.parquet")

In [66]:
score.to_pandas().to_parquet  ("../../Testing_Data/score_llama_answer_openAi_scoreOpenAi_2_DB.parquet")

In [94]:
#"Meta-Llama-3.1-8B-Instruct"
#chunksize, answer correctness, faithfulness, number of question
#10000, 0.592, 0.698, 50
# 5000, 0.617, 0.700, 50
# 3500, 0.677, 0.734, 50
# 3000, 0.688, 0.800, 50
# 2000, 0.629, 0.742, 50
# 1000, 0.644, 0.703, 50
#  500, 0.619, 0.503, 50
#  250, 0.594, 0.527, 50



In [ ]:
#2500, 0.62, 0.825
#3000, 0.60, 0.78 llama, llamaa, llama, llama
#3000, 062, 0.73, llama, llama, openAi, llama